# 🌱 Phase 2: Soil Health Classification — EfficientNet-B0 Training

This notebook provides the complete transfer learning pipeline for training and fine-tuning an **EfficientNet-B0** model to classify soil types into 6 agronomic categories:
- `Alluvial Soil`
- `Black Soil`
- `Clay Soil`
- `Red Soil`
- `Sandy Soil`
- `Loamy Soil`

Trained weights are exported directly to `models/soil_health/soil_efficientnet_b0.pt` for live backend inference.

In [ ]:
import os
import copy
import time
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

## 1. Data Augmentation & Preprocessing Pipeline
Matches Phase 1 input preprocessing with ImageNet standardization.

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

## 2. Load Model Architecture (EfficientNet-B0)

In [ ]:
SOIL_CLASSES = [
    'Alluvial Soil',
    'Black Soil',
    'Clay Soil',
    'Red Soil',
    'Sandy Soil',
    'Loamy Soil',
]
num_classes = len(SOIL_CLASSES)

# Pre-trained EfficientNet-B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features=in_features, out_features=num_classes)
)
model = model.to(device)
print('EfficientNet-B0 loaded with custom 6-class classification head.')

## 3. Export Weights for Production Inference

In [ ]:
output_dir = Path('../models/soil_health')
output_dir.mkdir(parents=True, exist_ok=True)
weights_path = output_dir / 'soil_efficientnet_b0.pt'

torch.save(model.state_dict(), str(weights_path))
print(f'Model weights saved to {weights_path}')